In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess

In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE

In [ ]:
from pyfuncs.io import ambient_fraction_per_gene
from pyfuncs.trajectory import (
    normalize_velocyto_layers,
    run_velocity,
    paga_report,
    paga_stability,
    plot_velocity,
    run_phate
)

from pyfuncs.interactions import (mouse_to_human, prepare_cpdb_input,
                                  download_cpdb_database, run_cpdb_statistical,
                                  filter_interactions, plot_interactions)

In [ ]:
DATA_DIR = f"{BASE_DIR}/data/ARAUZO_03/"

LEIDEN = dict(flavor="igraph", n_iterations=2, directed=False,
              random_state=SEED, neighbors_key="neighbors_harmony")

In [ ]:
GRUPO = "cell_type"     # la partición sobre la que se define PAGA
LOTE  = "gsm"

# Globinas: 45% del transcriptoma en algunas muestras, y 100% spliced.
# Fuera de velocity_genes sí o sí.
HB = ["Hbb-bs", "Hba-a1", "Hba-a2", "Hbb-bt", "Alas2", "Ahsp", "Bpgm"]

# Adata loading

In [ ]:
adata = sc.read(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed.h5ad")

In [ ]:
from pyfuncs.interactions import (liana_resources, resource_coverage,
                                  run_liana, filter_liana,
                                  compare_resources, plot_liana)

In [ ]:
liana_resources()                                  # ¿qué sirve hoy?
resource_coverage(["Il6","Il1b"], ["mouseconsensus"])   # el techo

In [ ]:
res = run_liana(adata, "cell_type", resource="mouseconsensus", n_perms=10000)
il  = filter_liana(res, ["Il6","Il1b"], max_rank=0.2)
il

In [ ]:
res

In [ ]:
plot_liana(adata, source_labels=["FAP.4", "FAP.5", "FAP.6"])

In [ ]:
sc.pl.umap(adata, color=["cell_type", "Il6",  "Il6st", "Il6ra"], cmap=magma)
sc.pl.umap(adata, color=["Il1r2", "Il1r1" ], cmap=magma)